In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
import joblib
from tqdm import tqdm
import time
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, LSTM, Conv1D, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModel
from concurrent.futures import ThreadPoolExecutor
import hashlib

In [ ]:
# ✅ Enable Full GPU Usage
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# ✅ Load ProtBERT Model & Tokenizer
protbert_tokenizer = AutoTokenizer.from_pretrained("Rostlab/prot_bert")
protbert_model = AutoModel.from_pretrained("Rostlab/prot_bert").to(device)

In [ ]:
# ✅ Dynamic Programming: Cache ProtBERT Embeddings
embedding_cache = {}

def hash_sequence(seq):
    return hashlib.md5(seq.encode()).hexdigest()

In [ ]:
# ✅ Modified Load Data Function to read from B2:B25001 and preserve order
def load_data(base_folder):
    sequences, labels = [], []
    category_counts = {}
    
    # Store all CSV filenames
    all_csv_files = [f for f in os.listdir(base_folder) if f.endswith(".csv")]
    
    # Sort to ensure consistent order for category labeling
    all_csv_files.sort()
    
    print("Reading datasets in the following order:")
    for idx, filename in enumerate(all_csv_files):
        # Use index+1 as the category label to ensure ordering
        category_id = idx + 1
        label = f"Category{category_id}"
        
        # Save original filename for reference
        original_name = filename.replace(".csv", "")
        print(f"{idx+1}. {filename} → {label} (Original: {original_name})")
        
        file_path = os.path.join(base_folder, filename)
        
        # Read only column B (index 1) starting from row 2
        try:
            df = pd.read_csv(file_path, usecols=[1], skiprows=1, header=None, nrows=25000)
            df.columns = ['Sequence']
            
            # Count sequences per category
            valid_sequences = df['Sequence'].dropna().tolist()
            category_counts[label] = len(valid_sequences)
            
            # Add sequences and labels
            for seq in valid_sequences:
                if isinstance(seq, str) and len(seq) > 0:
                    sequences.append(seq)
                    labels.append(label)
                    
            print(f"   Loaded {len(valid_sequences)} sequences from {label}")
        except Exception as e:
            print(f"   Error loading {file_path}: {e}")
    
    print("\nCategory distribution:")
    for category, count in category_counts.items():
        print(f"{category}: {count} sequences")
        
    # Store the category mapping for reference
    category_mapping = {f"Category{idx+1}": filename.replace(".csv", "") 
                        for idx, filename in enumerate(all_csv_files)}
    joblib.dump(category_mapping, "category_mapping.pkl")
    print("\nCategory mapping saved to category_mapping.pkl")
        
    return sequences, labels

In [ ]:
# ✅ Divide & Conquer: Parallel Feature Extraction
def extract_protbert_features(sequences):
    def process_sequence(seq):
        seq_hash = hash_sequence(seq)
        if seq_hash in embedding_cache:
            return embedding_cache[seq_hash]

        seq = ' '.join(list(seq))  
        encoded = protbert_tokenizer.batch_encode_plus(
            [seq], padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output = protbert_model(**encoded).last_hidden_state.mean(dim=1).cpu().numpy().flatten()

        embedding_cache[seq_hash] = output  
        return output

    with ThreadPoolExecutor() as executor:
        embeddings = list(tqdm(executor.map(process_sequence, sequences), total=len(sequences), desc="Extracting ProtBERT Features"))

    return np.array(embeddings, dtype=np.float32)

In [ ]:
# ✅ Load & Preprocess Data
base_folder = r"C:\\Users\\SURYA HA\\OneDrive\\Documents\\Prediction of Therapeutic Peptide using Deep Learning and DAA\\data\\Therapeutic Category Classification"
sequences, labels = load_data(base_folder)

# Check if we have data
if len(sequences) == 0:
    raise ValueError("No sequences were loaded. Please check your data files.")

print(f"Total sequences loaded: {len(sequences)}")
print(f"Unique categories: {len(set(labels))}")

# Extract features
protbert_features = extract_protbert_features(sequences)
scaler = StandardScaler()
X = scaler.fit_transform(protbert_features)
joblib.dump(scaler, "scaler 2.pkl")

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
joblib.dump(label_encoder, "label_encoder.pkl")

In [ ]:
# ✅ Feature Selection using PCA
pca = PCA(n_components=min(50, len(sequences) - 1))  # Ensure n_components is valid
X_pca = pca.fit_transform(X)
joblib.dump(pca, "pca_model.pkl")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42, stratify=y)
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

In [ ]:
# ✅ CNN-LSTM Model (Efficient)
num_classes = len(set(labels))
input_shape = (X_train.shape[1], 1)

with tf.device('/GPU:0'):
    model = Sequential([
        Conv1D(32, 3, activation='relu', input_shape=input_shape),
        MaxPooling1D(2),
        LSTM(64),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
    
    # Add class weights to handle imbalanced data
    class_weights = {}
    class_counts = np.bincount(y_train)
    total = len(y_train)
    for i in range(len(class_counts)):
        class_weights[i] = total / (len(class_counts) * class_counts[i])
    
    # Train with class weights
    history = model.fit(
        X_train, y_train, 
        epochs=40, 
        batch_size=32, 
        validation_data=(X_test, y_test),
        class_weight=class_weights
    )

model.save("model 2.h5")

In [ ]:
# ✅ Accuracy Evaluation
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Generate classification report
from sklearn.metrics import classification_report, confusion_matrix
y_pred = np.argmax(model.predict(X_test), axis=1)
print("\nClassification Report:")
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
print(report)

In [ ]:
# ✅ Plot confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

In [ ]:
# ✅ Modified Confidence Adjustment with 0.3 threshold
def confidence_adjustment(prediction):
    max_prob = np.max(prediction)
    if max_prob < 0.3:  # Changed threshold from 0.7 to 0.3
        return "Unknown Category"
    return label_encoder.inverse_transform([np.argmax(prediction)])[0]